# NSL-KDD Intrusion Detection with XAI and LLMs

In [1]:
!pip install shap lime -q


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from lime import lime_tabular
from openai import OpenAI

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


## Configuration

In [3]:
RANDOM_STATE = 42
TEST_SIZE = 0.30
SHAP_SAMPLE_SIZE = 200

DATASET_PATH = (
    "../datasets/dataset3/"
    "KDDTrain+_20Percent.txt"
)


## Dataset Loading

In [4]:
df = pd.read_csv(
    DATASET_PATH,
    header=None
)

print(df.shape)

df.head()


(25192, 43)


,0,1,2,3,4,5,6,7,8,9,...,33,34,35,36,37,38,39,40,41,42
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


## Features naming

In [5]:
# Add the names of the features/columns
columns = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in",
    "num_compromised","root_shell","su_attempted","num_root",
    "num_file_creations","num_shells","num_access_files",
    "num_outbound_cmds","is_host_login","is_guest_login","count",
    "srv_count","serror_rate","srv_serror_rate","rerror_rate",
    "srv_rerror_rate","same_srv_rate","diff_srv_rate",
    "srv_diff_host_rate","dst_host_count","dst_host_srv_count",
    "dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate",
    "dst_host_serror_rate","dst_host_srv_serror_rate",
    "dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label",
    "difficulty_level"
]

df.columns = columns

In [6]:
for col in ['protocol_type', 'service', 'flag']:
    df[col] = df[col].astype('category').cat.codes


## Target Distribution

In [7]:
# Plot a distribution graph of the Target (y) "attack_detected"
plt.figure(figsize=(6, 4))
sns.countplot(x='label', hue='label', data=df, palette='viridis', legend=False)
plt.title('Distribuição da variável \"label (0: Normal | 1: Anormal)\"')
plt.xlabel('label')
plt.ylabel('Contagem')
plt.show()

print(df['label'].value_counts(normalize=True).round(3))

label
normal             0.534
neptune            0.329
ipsweep            0.028
satan              0.027
portsweep          0.023
smurf              0.021
nmap               0.012
back               0.008
teardrop           0.007
warezclient        0.007
pod                0.002
guess_passwd       0.000
warezmaster        0.000
buffer_overflow    0.000
imap               0.000
rootkit            0.000
multihop           0.000
phf                0.000
ftp_write          0.000
land               0.000
loadmodule         0.000
spy                0.000
Name: proportion, dtype: float64


/tmp/ipykernel_1067353/2913597791.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Train/Test Split

In [8]:
X = df.drop(['label','difficulty_level'], axis=1)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)


## Random Forest Training

In [9]:
rf_model = RandomForestClassifier(
    random_state=RANDOM_STATE
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    y_pred
)

print(f"Accuracy: {accuracy:.4f}")

print(
    classification_report(
        y_test,
        y_pred
    )
)


Accuracy: 0.9971
                 precision    recall  f1-score   support

           back       1.00      0.98      0.99        64
buffer_overflow       1.00      1.00      1.00         2
   guess_passwd       1.00      1.00      1.00         3
           imap       1.00      0.50      0.67         2
        ipsweep       1.00      1.00      1.00       206
     loadmodule       0.00      0.00      0.00         1
        neptune       1.00      1.00      1.00      2523
           nmap       0.99      1.00      0.99        80
         normal       1.00      1.00      1.00      4042
            phf       0.00      0.00      0.00         1
            pod       1.00      1.00      1.00        12
      portsweep       0.99      0.99      0.99       170
        rootkit       0.00      0.00      0.00         0
          satan       0.99      0.97      0.98       194
          smurf       1.00      1.00      1.00       164
            spy       0.00      0.00      0.00         1
       teardr

/home/beloin/Documents/agents/Can-LLMs-Explain-AI-A-Study-of-XAI-in-Cybersecurity-ML-Models/venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/beloin/Documents/agents/Can-LLMs-Explain-AI-A-Study-of-XAI-in-Cybersecurity-ML-Models/venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/beloin/Documents/agents/Can-LLMs-Explain-AI-A-Study-of-XAI-in-Cybersecurity-ML-Models/venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Pre

## Fuzzy Matrix

In [10]:
# Get and reshape confusion matrix data
matrix = confusion_matrix(y_test, y_pred)
matrix = matrix.astype('float') / matrix.sum(axis=1)[:, np.newaxis]

# Build the plot
plt.figure(figsize=(6,3))
sns.set(font_scale=1.4)
sns.heatmap(matrix, annot=True, annot_kws={'size':10},
            cmap=plt.cm.Reds, linewidths=0.2)

# Add labels to the plot
class_names = ['Normal', 'Abnormal']
tick_marks = np.arange(len(class_names))
tick_marks2 = tick_marks + 0.5
plt.xticks(tick_marks, class_names, rotation=25)
plt.yticks(tick_marks2, class_names, rotation=0)
plt.xlabel('Prediction')
plt.ylabel('Real')
plt.title('Model [Dataset 3] Fuzzy Matrix')
plt.show()

/tmp/ipykernel_1067353/1639875216.py:3: RuntimeWarning: invalid value encountered in divide
  matrix = matrix.astype('float') / matrix.sum(axis=1)[:, np.newaxis]


/tmp/ipykernel_1067353/1639875216.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## SHAP Explainability

In [11]:
sample_idx = np.random.choice(
    X_test.index,
    size=min(
        SHAP_SAMPLE_SIZE,
        len(X_test)
    ),
    replace=False
)

X_sample = X_test.loc[sample_idx]

explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_sample)

shap.summary_plot(
    [
        shap_values[:, :, 0],
        shap_values[:, :, 1]
    ],
    X_sample,
    plot_type="bar",
    class_names=["Normal", "Abnormal"],
    max_display=9,
    show=False
)

plt.legend(
    loc='lower right'
)

plt.show()


/home/beloin/Documents/agents/Can-LLMs-Explain-AI-A-Study-of-XAI-in-Cybersecurity-ML-Models/venv/lib/python3.14/site-packages/shap/plots/_beeswarm.py:1150: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
/tmp/ipykernel_1067353/3682422153.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## LIME Explainability

In [12]:
feature_names = list(X_test)
class_names = list(y_test.unique())

# Creates a LIME object for explainability (Local Explainability)
# LIME needs access to the training data and the names of the target features.
explainer_l = lime_tabular.LimeTabularExplainer(
    X_test.values,  # Convert DataFrame to NumPy array
    feature_names=feature_names,
    class_names=class_names,
    mode='classification',
    verbose=True
)

# Choose any instance of the data to generate the explainability (Ex: 100)
i = 50
instance_to_explain = X_test.iloc[i].values
# Generates the explanation for the chosen instance
# The predict_proba function of the model is passed to LIME
exp = explainer_l.explain_instance(
    instance_to_explain,
    rf_model.predict_proba,
    num_features=9  # Number of features to be considered for the explanation.
)

# Viewing the explanation via LIME
actual_class_label = y_test.iloc[i]
predicted_class_label = rf_model.predict(instance_to_explain.reshape(1, -1))[0]

print(f"Classificação real: {actual_class_label}")
print(f"Classificação prevista: {predicted_class_label}")

print("\nExplanation details:")
print(exp.as_list())


Intercept 0.017988933705376003
Prediction_local [0.00290311]
Right: 0.0
Classificação real: neptune
Classificação prevista: neptune

Explanation details:
[('src_bytes <= 0.00', 0.007790575947725274), ('logged_in <= 0.00', -0.005430223152161079), ('num_failed_logins <= 0.00', -0.005152544185408844), ('num_access_files <= 0.00', -0.004049986160842337), ('hot <= 0.00', -0.003924120694226553), ('dst_bytes <= 0.00', -0.003430192026405102), ('num_shells <= 0.00', 0.003206148040027843), ('num_root <= 0.00', -0.002235754638633795), ('0.05 < dst_host_same_srv_rate <= 0.51', -0.001859728200336936)]


/home/beloin/Documents/agents/Can-LLMs-Explain-AI-A-Study-of-XAI-in-Cybersecurity-ML-Models/venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/home/beloin/Documents/agents/Can-LLMs-Explain-AI-A-Study-of-XAI-in-Cybersecurity-ML-Models/venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


## TesteLLM Context Information

In [13]:
# ==========================
# COLUMN DESCRIPTION
# ==========================
column_description = {
    "duration": "Duração da conexão em segundos",
    "protocol_type": "Protocolo da conexão, podendo ser TCP, UDP ou ICMP",
    "service": "Tipo de serviço de rede acessado (ex: http, ftp, smtp)",
    "flag": "Estado da conexão (indicador de sucesso, rejeição, reset, etc.)",
    "src_bytes": "Quantidade de bytes enviados pelo host de origem",
    "dst_bytes": "Quantidade de bytes enviados pelo host de destino",
    "land": "Indica se a conexão é do tipo land (mesmo IP e porta de origem e destino): sim (1) ou não (0)",
    "wrong_fragment": "Número de fragmentos de pacotes incorretos",
    "urgent": "Número de pacotes com flag urgente ativada",
    "hot": "Número de eventos ‘hot’ na conexão (acessos sensíveis ou suspeitos)",
    "num_failed_logins": "Número de tentativas de login que falharam",
    "logged_in": "Indica se o usuário conseguiu logar na máquina: sim (1) ou não (0)",
    "num_compromised": "Número de arquivos comprometidos na máquina alvo",
    "root_shell": "Indica se foi obtido acesso root: sim (1) ou não (0)",
    "su_attempted": "Indica se houve tentativa de uso do comando 'su' para troca de usuário",
    "num_root": "Número de acessos com privilégios de root",
    "num_file_creations": "Número de operações de criação de arquivos",
    "num_shells": "Número de shells abertos durante a sessão",
    "num_access_files": "Número de operações de acesso a arquivos (ex: leitura/escrita)",
    "num_outbound_cmds": "Número de comandos outbound em uma sessão FTP (normalmente 0 no dataset)",
    "is_host_login": "Indica se o login pertence a uma lista de hosts confiáveis: sim (1) ou não (0)",
    "is_guest_login": "Indica se o login foi feito como convidado (guest): sim (1) ou não (0)",
    "count": "Número de conexões do mesmo tipo na última janela de tempo",
    "srv_count": "Número de conexões do mesmo serviço na última janela de tempo",
    "serror_rate": "Proporção de conexões com erros de TCP em relação ao total",
    "srv_serror_rate": "Proporção de conexões com erros de TCP no mesmo serviço",
    "rerror_rate": "Proporção de conexões com erros de protocolo de rede",
    "srv_rerror_rate": "Proporção de conexões com erros de protocolo no mesmo serviço",
    "same_srv_rate": "Proporção de conexões para o mesmo serviço",
    "diff_srv_rate": "Proporção de conexões para serviços diferentes",
    "srv_diff_host_rate": "Proporção de conexões para o mesmo serviço em diferentes hosts",
    "dst_host_count": "Número de conexões para o mesmo host de destino",
    "dst_host_srv_count": "Número de conexões para o mesmo serviço no host de destino",
    "dst_host_same_srv_rate": "Proporção de conexões para o mesmo serviço no host de destino",
    "dst_host_diff_srv_rate": "Proporção de conexões para serviços diferentes no host de destino",
    "dst_host_same_src_port_rate": "Proporção de conexões para o host de destino usando a mesma porta de origem",
    "dst_host_srv_diff_host_rate": "Proporção de conexões para o mesmo serviço em diferentes hosts de destino",
    "dst_host_serror_rate": "Proporção de conexões com erros TCP para o host de destino",
    "dst_host_srv_serror_rate": "Proporção de conexões com erros TCP no mesmo serviço para o host de destino",
    "dst_host_rerror_rate": "Proporção de conexões com erros de protocolo para o host de destino",
    "dst_host_srv_rerror_rate": "Proporção de conexões com erros de protocolo no mesmo serviço para o host de destino",
    "label": "Classe da conexão, podendo ser: normal (0) ou anormal (1)"
}

# ==========================
# CATEGORY_ENCODING
# ==========================
category_encoding = {
    "protocol_type": {
        "ICMP": 0,
        "TCP": 1,
        "UDP": 2
    },

    "service": {
        "IRC": 0,
        "X11": 1,
        "Z39_50": 2,
        "auth": 3,
        "bgp": 4,
        "courier": 5,
        "csnet_ns": 6,
        "ctf": 7,
        "daytime": 8,
        "discard": 9,
        "domain": 10,
        "domain_u": 11,
        "echo": 12,
        "eco_i": 13,
        "ecr_i": 14,
        "efs": 15,
        "exec": 16,
        "finger": 17,
        "ftp": 18,
        "ftp_data": 19,
        "gopher": 20,
        "hostnames": 21,
        "http": 22,
        "http_443": 23,
        "http_8001": 24,
        "imap4": 25,
        "iso_tsap": 26,
        "klogin": 27,
        "kshell": 28,
        "ldap": 29,
        "link": 30,
        "login": 31,
        "mtp": 32,
        "name": 33,
        "netbios_dgm": 34,
        "netbios_ns": 35,
        "netbios_ssn": 36,
        "netstat": 37,
        "nnsp": 38,
        "nntp": 39,
        "ntp_u": 40,
        "other": 41,
        "pm_dump": 42,
        "pop_2": 43,
        "pop_3": 44,
        "printer": 45,
        "private": 46,
        "red_i": 47,
        "remote_job": 48,
        "rje": 49,
        "shell": 50,
        "smtp": 51,
        "sql_net": 52,
        "ssh": 53,
        "sunrpc": 54,
        "supdup": 55,
        "systat": 56,
        "telnet": 57,
        "tim_i": 58,
        "time": 59,
        "urh_i": 60,
        "urp_i": 61,
        "uucp": 62,
        "uucp_path": 63,
        "vmnet": 64,
        "whois": 65
    },

    "flag": {
        "OTH": 0,
        "REJ": 1,
        "RSTO": 2,
        "RSTOS0": 3,
        "RSTR": 4,
        "S0": 5,
        "S1": 6,
        "S2": 7,
        "S3": 8,
        "SF": 9,
        "SH": 10
    },

    "label": {
        "normal": 0,
        "anormal": 1
    }
}


# ==========================
# TRAINING DATA SAMPLE
# ==========================
train_sample = X_train.sample(50)
train_sample["label"] = y_train.loc[train_sample.index]
train_sample_json = train_sample.to_json(orient="records")

# ==========================
# PREDICTION SAMPLE
# ==========================
pred_sample = pd.DataFrame({
    "real": y_test,
    "predicted": y_pred
})

pred_sample_json = pred_sample.sample(50).to_json(orient="records")

# ==========================
# MODEL INFORMATION
# ==========================
model_info = {
"model_type": "Random Forest",
"task": "Intrusion detection from log. Each log entry is classified as: Normal (0) or Abnormal (1)",
"target_variable": "label",
"features": list(X.columns)
}

## Prompt Construction

In [14]:
# ============================================================
# Prompt Construction
# ============================================================

prompt = f"""
You are an expert in Explainable Artificial Intelligence (XAI)
and Cybersecurity.

Analyze the machine learning model and provide a clear,
technical, and objective explanation of its behavior.

=========================
MODEL INFORMATION
=========================
{model_info}

=========================
COLUMN DESCRIPTION
=========================
{column_description}

=========================
CATEGORY ENCODING
=========================
{category_encoding}

=========================
TRAINING DATA SAMPLE
=========================
{train_sample_json}

=========================
PREDICTION SAMPLE
=========================
{pred_sample_json}

=========================
TASK
=========================

1. Identify the top-3 most relevant features for each class.
2. Compare differences between classes.
3. Interpret model behavior in the cybersecurity context.
4. Avoid causal claims — describe only associations.

The explanation should be understandable for both technical
and non-technical users.
"""


## GPT-5 Explainability

In [15]:
# ============================================================
# GPT-5 Explainability
# ============================================================

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    timeout=300,
)

response = client.responses.create(
    model="gpt-5",
    input=[
        {
            "role": "system",
            "content": (
                "You are an expert in Machine Learning "
                "and Explainable AI."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    max_output_tokens=12288
)

## Extract LLM Explanation


In [16]:
# ============================================================
# Extract LLM Explanation
# ============================================================

explanation = ""

for item in response.output:
    if item.type == "message":
        for content in item.content:
            if content.type == "output_text":
                explanation += content.text

print("\n===== MODEL EXPLANATION =====\n")
print(explanation)



===== MODEL EXPLANATION =====

Summary and scope
- There is a mismatch between the stated task (binary: normal vs abnormal) and the data/predictions (multiclass with attack types such as neptune, smurf, portsweep, back, teardrop, ipsweep, satan). The explanation below covers both:
  1) the high‑level Normal vs Attack separation the forest appears to learn, and
  2) per‑attack class behavior, based on patterns present in the provided training and prediction samples.
- All statements below describe associations the trained Random Forest tends to use; they are not causal claims.

1) Top‑3 features per class (observed)
Normal
- flag (connection state): The model associates normal traffic with successful/established flags (e.g., SF).
- serror_rate and srv_serror_rate (TCP SYN error proportions): Near 0 for normal sessions in the sample.
- logged_in: Normal application use often has logged_in=1; many attacks show logged_in=0.

neptune (TCP SYN flood DoS)
- serror_rate and srv_serror_rate: T

## Local LLM Support


In [17]:
# ============================================================
# Local LLM Integration
# ============================================================

# Adapt this section to test local/open-source LLMs
# such as:
#
# - GPT-OSS-20B
# - Llama 3
# - Mistral
# - DeepSeek
#
# Example frameworks:
# - Ollama
# - vLLM
# - LM Studio
# - Transformers


In [18]:
# ============================================================
# Local LLM via Ollama (OpenAI-compatible endpoint)
# ============================================================

local_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
    timeout=300
)

local_response = local_client.responses.create(
    model="gpt-oss:20b-cloud",
    input=[
        {
            "role": "system",
            "content": (
                "You are an expert in Machine Learning "
                "and Explainable AI."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    max_output_tokens=12288
)

local_explanation = ""

for item in local_response.output:
    if item.type == "message":
        for content in item.content:
            if content.type == "output_text":
                local_explanation += content.text

print("\n===== LOCAL LLM EXPLANATION =====\n")
print(local_explanation)



===== LOCAL LLM EXPLANATION =====

**Explainable Insight into the Random Forest Intrusion‑Detection Model**  
*(technical audience – simplified for non‑technical readers in brackets)*  

---

## 1. Feature importance – what the model “tells” us

Random‑Forest learns by splitting the data into many decision trees.  
Each time a tree uses a feature to make a split, that feature “explains” a portion of the uncertainty (impurity) in the data.  
The importance of a feature is simply the total impurity reduction it brings across all trees, divided by the number of trees.

Because the model was trained on 71 features, the ones that consistently reduce uncertainty the most show up in the rankings.  
Below are the **top 3 features that most influence the model’s prediction for each major class** that appears in the test predictions:

| Class | Top 3 features (ordered by importance) | What it means (short note) |
|-------|---------------------------------------|----------------------------|
| *